# ⚖️ Comparativa de Modelos: Base vs. Fine-Tuned

Este notebook permite evaluar visualmente las diferencias en las respuestas entre el modelo original (base) y el modelo entrenado con tus datos específicos.

In [ ]:
import os
import json
import pandas as pd
from dotenv import load_dotenv
from openai import AzureOpenAI
from IPython.display import display, HTML

# Cargar configuración
load_dotenv(dotenv_path="../.env")

client = AzureOpenAI(
    api_key=os.getenv("Key"),
    api_version="2024-05-01-preview",
    azure_endpoint=os.getenv("base_url").replace("/openai/v1", "")
)

base_model = os.getenv("BASE_MODEL_DEPLOYMENT")
ft_model = os.getenv("FINETUNED_MODEL_DEPLOYMENT")

print(f"📌 Modelo Base: {base_model}")
print(f"📌 Modelo Fine-Tuned: {ft_model}")

## 1. Preparación de Casos de Prueba
Vamos a extraer algunos ejemplos del set de validación para ver cómo se comportan los modelos.

In [ ]:
def get_test_cases(file_path, num_samples=3):
    test_cases = []
    with open(file_path, 'r', encoding='utf-8') as f:
        for i, line in enumerate(f):
            if i >= num_samples: break
            data = json.loads(line)
            # Extraemos el mensaje del sistema y el del usuario
            system_msg = data['messages'][0]['content']
            user_msg = data['messages'][1]['content']
            expected = data['messages'][2]['content']
            test_cases.append({"system": system_msg, "user": user_msg, "expected": expected})
    return test_cases

test_cases = get_test_cases("validation_set.jsonl")
print(f"✅ {len(test_cases)} casos cargados para la comparativa.")

## 2. Ejecución de la Comparativa
Enviaremos las mismas preguntas a ambos modelos y guardaremos las respuestas.

In [ ]:
results = []

for case in test_cases:
    # Respuesta Modelo Base
    try:
        resp_base = client.chat.completions.create(
            model=base_model,
            messages=[{"role": "system", "content": case['system']}, {"role": "user", "content": case['user']}],
            max_tokens=60
        ).choices[0].message.content
    except Exception as e: resp_base = f"Error: {e}"

    # Respuesta Modelo Fine-Tuned
    try:
        resp_ft = client.chat.completions.create(
            model=ft_model,
            messages=[{"role": "system", "content": case['system']}, {"role": "user", "content": case['user']}],
            max_tokens=60
        ).choices[0].message.content
    except Exception as e: resp_ft = f"Error: {e}"

    results.append({
        "Input": case['user'],
        "Base Model Response": resp_base,
        "Fine-Tuned Response": resp_ft,
        "Expected (Ground Truth)": case['expected']
    })

df = pd.DataFrame(results)
display(HTML("<h3>Resultados de la Comparativa</h3>"))
display(df)

## 3. Conclusión
Observa si el modelo Fine-Tuned sigue mejor el formato esperado o si sus predicciones son más precisas comparadas con el *Expected* (Ground Truth).